# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [ ]:
%uv pip install -q "transformers>=4.51.0" accelerate safetensors hf_transfer

In [2]:
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is recommended for this 4B model.")

model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print("GPU:", torch.cuda.get_device_name(0))
print("Loading:", MODEL_ID)
print("Dtype:", model_dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=model_dtype,
    device_map="auto",
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
model.eval()

print("Model ready on:", model.device)

GPU: NVIDIA L40S
Loading: Qwen/Qwen3-4B-Instruct-2507
Dtype: torch.bfloat16


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model ready on: cuda:0


In [20]:
NO_ANSWER = "සපයා ඇති සන්දර්භයේ පිළිතුර නොමැත."

SYSTEM_PROMPT = """You are a helpful Sinhala history question-answering assistant.

Your task is to answer the question using ONLY the information explicitly provided in the context.

Instructions:

- Read the entire context carefully before answering.
- Use only the information explicitly stated in the context.
- Do not use external knowledge, assumptions, or prior knowledge.
- Identify the exact information requested by the question.
- If the answer is found in multiple parts of the context, combine the relevant information into a single complete answer.
- Include only information that directly answers the question.
- Do not include additional facts, names, dates, or events unless they are required to answer the question.
- Match the person or entity named in the question exactly.
- Use evidence that contains both the requested entity and the requested attribute.
- Do not take a date or fact from a neighboring sentence about a different entity or event.
- Do not infer or guess information that is not explicitly stated, except for simple arithmetic explicitly requested by the question when all required values are stated in the context.
- For a duration question with explicit starting and ending years, subtract the starting year from the ending year and return the duration.
- If the answer cannot be found in the context, respond exactly with:
  \"සපයා ඇති සන්දර්භයේ පිළිතුර නොමැත.\"
- Return only the final answer in natural Sinhala.
- Do not explain your reasoning.
- Do not mention passage numbers, page numbers, chapter names, grades, or any other source references."""


MAX_INPUT_TOKENS = 16384


def ask_from_context(context, question, max_new_tokens=64, debug=False):
    user_prompt = f"""Context:

{context}

Question:

{question}

Answer:"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    rendered_prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    input_token_count = inputs["input_ids"].shape[-1]
    if input_token_count > MAX_INPUT_TOKENS:
        raise ValueError(
            f"Prompt has {input_token_count:,} tokens, exceeding the evaluation limit "
            f"of {MAX_INPUT_TOKENS:,}. Retrieve or rerank fewer context passages."
        )

    if debug:
        print("=== Rendered prompt tail ===")
        print(rendered_prompt[-2000:])
        print("=== Inference settings ===")
        print("Input tokens:", input_token_count)
        print("do_sample: False")
        print("max_new_tokens:", max_new_tokens)
        print("eos_token_id:", model.generation_config.eos_token_id)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=model.generation_config.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return answer


print("ask_from_context() is ready")

ask_from_context() is ready


In [26]:
context = """[රූපය ජෝන් පයිබස් කීර්ති ශ්‍රී රාජසිංහ රජු හමු වූ අවස්ථාව නූතන චිත්‍ර ශිලිපියෙකුගේ ඇසින් (චරිත් චතුරං. මහතාගේ අනුග්‍රහයෙනි.) එහි ප්‍රතිඵලයක් ලෙස උඩරට රාජ්‍යය හා ඉංග්‍රීසින් අතර ප්‍රධාන ත ගමන් කිහිපයක් සිදු විය.

1782 වර්ෂයේ දී හියු බොයිඩ් රාජාධිරාජසිංහ රජු හමු වීම][ී ජෝන් පයිබස් කීර්ති ශ්‍රී රාජසිංහ රජු හමු වීම

1782 වර්ෂයේ දී හියු බොයිඩ් රාජාධිරාජසිංහ රජු හමු වීම1795 වර්ෂයේ දී රොබට් ඇන්ඩෲස් රාජාධිරාජසිංහ රජු හමු වීම ඉංග්‍රීසී තයන් මෙරටට පැමිණි මුල් කාලයේ දී ලන්දේසීන් හා ඉංග්‍රීසීන් අතර මිත්‍ර සබඳතාවක් පැවතුණි. ඒ නිසා ලන්දේසීන් සම. ගැටුම් ඇති කර ගැනීමට ඉංග්‍රීසීහු අකමැති වූහ. එහෙයින් මුල් ත ගමන් දෙකේ දී උඩරට රජුන්ගේ අරමුණු ඉටු නොවුණි. රොබට් ඇන්ඩෲස් 1795 වර්ෂයේ දී රාජාධිරාජසිංහ රජු හමු වූ අවස්ථාවේ දී ඉංග්‍රීසින් ලන්දේසී බලකොටු යටත් කිරීමේ දී රජුගේ සහාය ඉල්ලා සිටියේ ය. රජු ඊට කැමති වුවත් ලන්දේසි බලකොටු යටත් කිරීම ඉංග්‍රීසින් අපේක්ෂා කළ තරම් දුෂ්කර නොවූ හෙයින් ඒ සඳහා රජුගේ සහාය ලබා ගැනීමට ද ඔවුහු ඉදිරිපත් නොවූහ. පෙරදි. ඉන්දියා ඉංග්‍රීසි වෙළෙඳ සමාගම මෙරට මුහුදුබඩ ප්‍රදේශවල බලය පිහිටුවීම ඉංග්‍රීසින්ට පෙර මෙරට මුහුදුබඩ ප්‍රදේශ අල්ලාගෙන සිටි ලන්දේසීන්ගේ මව්රට වූයේ ඕලන්දයයි.][්‍රී ලංකාවේ පිහිටීම කුරුඳු ඇතුළු වටිනා කුළුබඩු ලබාගත හැකි වීම ඉංග්‍රීසීන් හා උඩරට රාජ්‍ය අතර ත සබඳතාපෘතුගීසීන් මෙරටින් පලවා හැරීමට ලන්දේසින්ගේ සහය ලබා ගත් සෙයින් ම ලන්දේසීන් දිවයිනෙන් පලවා හැරීම සඳහා වෙනත් විදේශ ජාතියක සහාය ලබා ගැනීමට උඩරට රජවරු කල්පනා කළහ. මේ අනුව කීර්ති ශ්‍රී රාජසිංහ රජතුමා විසින් ඉන්දියාවේ සිටි ඉංග්‍රීසීන් වෙත මෙන් ම ප්‍රංශවරුන් වෙත ද තයන් යවන ලදි. ඉංග්‍රීසි තයන් උඩරටට පැමිණීමට පසුබිම සැකසුණේ එහි ප්‍රතිඵලයක් වශයෙනි. ඉංග්‍රීසීන් මෙරට මුහුදුබඩ ප්‍රදේශ අත්පත් කර ගැනීමට පෙර ඔවුන්ගේ තයන් තිදෙනෙකු වරින් වර උඩරටට පැමිණ තිබේ. 1750 සිට 1775 දක්වා කීර්ති ශ්‍රී රාජසිංහ රජු පොලොන්නරුවේ රජ කලේය. 1762 දී ජෝන් පයිබස් උඩරටට පැමිණීම 1782 දී හියු බොයිඩ් උඩරටට පැමිණීම 1795 දී රොබට් ඇන්ඩෲස් උඩරටට පැමිණීම][ාංක හයෙන් එකකටවත් මහනුවරට ළඟා වීමට ඉඩ නොලැබුණි. සුපුරුදු ප්‍රහාර මඟින් උඩරැටියෝ මේ සේනාංක පරාජය කළහ. වර්ෂ 1765 දී ලන්දේසීිහු නැවත වරක් උඩරට ආක්‍රමණය කළහ. උඩරැටියන්ගේ ප්‍රහාරවලට මුහුණ දෙමින් මෙවර ලන්දේසීහු මහනුවරට ඇතුළු වීමට සමත් වූහ. කීර්ති ශ්‍රී රාජසිංහ රජු සාමයක් ඇති කර ගැනීමට කැමැත්ත දැක්වූ හෙයින් ලන්දේසීහු තමන්ට වාසිදායක කොන්දේසි මාලාවක් ඉදිරිපත් කළහ. එහෙත් ලන්දේසීන් ඉදිරිපත් කළ අසාධාරණ කොන්දේසි පිළිගැනීමට රජු අකමැති විය. මේ නිසා නගරය කොල්ල කෑ ලන්දේසීහු මාසයකට වැඩි කලක් එහි ගත කළහ. එහෙත් ආහාර හිඟය, ලෙඩ රෝ. හා වැසි සමය ආරම්භ වීම නිසා ලන්දේසීහු මහනුවර අතහැර නැවත කොළඹට පැමිණියහ. ලන්දේසීන්ගේ ආක්‍රමණ, තර්ජන හා වසර කිහිපයක් ඇදී ගිය යුද්ධවලින් පීඩා විඳීම නිසා වර්ෂ 1766 දී කීර්ති ශ්‍රී රාජසිංහ රජු ලන්දේසීන් සම. සාම ගිවිසුමක් අත්සන් කළේය. 1766 ගිවිසුම නිසා උඩරට රාජධානියට වෙරළබඩ තීිරයක් අහිමි විය. උඩරට සාම්ප්‍රදායික වෙළෙඳාමට ද මෙමගින් පහර වැදුණි.][පයක් ඇති විය. එම ගැටුම් නිම වූයේ වර්ෂ 1766 දී රජතුමා හා ලන්දේසීන් අතර සාම ගිවිසුමක් අත්සන් කිරීමෙනි. කීර්ති ශ්‍රී රාජසිංහ රජු ද උඩරට සිංහාසනයට උරුමකරුවෙකු නොතබා වර්ෂ 1781 දී මිය ගියේ ය. රාජාධිරාජසිංහ රජු (ක්‍රි.ව.1781 - 1798) කීර්ති ශ්‍රී රාජසිංහ රජුගෙන් පසු ඔහුගේ සහෝදරයා රාජාධිරාජසිංහ නමින් උඩරට රජ බවට පත්විය. මේ වන විට උඩරට රදල ප්‍රභූවරුන් හා නායක්කර් වංශික රජවරුන් අතර වූ මත භේද වර්ධනය වෙමින් තිබුණි. මෙම රාජ්‍ය සමයේ දී සිදු වූ වැදගත් සිදු වීමක් වූයේ ක්‍රිස්තු වර්ෂ 1796 දී ඉංග්‍රීසීන් ලංකාවේ වෙරළබඩ ප්‍රදේශ අල්ලා ගැනීමයි. රාජාධිරාජසිංහ රජු ද අනුප්‍රාප්තිකයෙකු නොතබා ම වර්ෂ 1798 දී මිය ගියේ ය. ශ්‍රි වික්‍රම රාජසිංහ රජු (ක්‍රි.ව.1798-1815).

1763 වර්ෂයේ දී ජෝන් පයිබස් කීර්ති ශ්‍රී රාජසිංහ රජු හමු වීම.]"""

question = "විජය රජු වසර කීයක් පොළොන්නරුවේ රජකම් කලාද?"

answer = "Single-example inference skipped; run the full test evaluation cell below."

print("Question:", question)
print("Answer:", answer)

=== Rendered prompt tail ===
්‍රීසි තයන් උඩරටට පැමිණීමට පසුබිම සැකසුණේ එහි ප්‍රතිඵලයක් වශයෙනි. ඉංග්‍රීසීන් මෙරට මුහුදුබඩ ප්‍රදේශ අත්පත් කර ගැනීමට පෙර ඔවුන්ගේ තයන් තිදෙනෙකු වරින් වර උඩරටට පැමිණ තිබේ. 1750 සිට 1775 දක්වා කීර්ති ශ්‍රී රාජසිංහ රජු පොලොන්නරුවේ රජ කලේය. 1762 දී ජෝන් පයිබස් උඩරටට පැමිණීම 1782 දී හියු බොයිඩ් උඩරටට පැමිණීම 1795 දී රොබට් ඇන්ඩෲස් උඩරටට පැමිණීම][ාංක හයෙන් එකකටවත් මහනුවරට ළඟා වීමට ඉඩ නොලැබුණි. සුපුරුදු ප්‍රහාර මඟින් උඩරැටියෝ මේ සේනාංක පරාජය කළහ. වර්ෂ 1765 දී ලන්දේසීිහු නැවත වරක් උඩරට ආක්‍රමණය කළහ. උඩරැටියන්ගේ ප්‍රහාරවලට මුහුණ දෙමින් මෙවර ලන්දේසීහු මහනුවරට ඇතුළු වීමට සමත් වූහ. කීර්ති ශ්‍රී රාජසිංහ රජු සාමයක් ඇති කර ගැනීමට කැමැත්ත දැක්වූ හෙයින් ලන්දේසීහු තමන්ට වාසිදායක කොන්දේසි මාලාවක් ඉදිරිපත් කළහ. එහෙත් ලන්දේසීන් ඉදිරිපත් කළ අසාධාරණ කොන්දේසි පිළිගැනීමට රජු අකමැති විය. මේ නිසා නගරය කොල්ල කෑ ලන්දේසීහු මාසයකට වැඩි කලක් එහි ගත කළහ. එහෙත් ආහාර හිඟය, ලෙඩ රෝ. හා වැසි සමය ආරම්භ වීම නිසා ලන්දේසීහු මහනුවර අතහැර නැවත කොළඹට පැමිණියහ. ලන්දේසීන්ගේ ආක්‍රමණ, තර්ජන හා වසර කිහිපයක් 

In [ ]:
import json
import re
import time
import unicodedata
from pathlib import Path

TEST_PATH = Path("/tmp/test_updated.jsonl")
RESULTS_PATH = Path("/tmp/qwen_test_updated_results.jsonl")
MAX_NEW_TOKENS = 64
LIMIT = None  # Keep as None to evaluate every record.


def normalize_answer(value):
    text = unicodedata.normalize("NFC", str(value))
    text = text.replace("\u200b", "").replace("\ufeff", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text.strip(" \t\r\n[]{}()<>\"'`.,!?;:।෴").casefold()


def reference_answer(item):
    if item.get("answerable") is False:
        return NO_ANSWER

    answer = item.get("answer", "")
    if isinstance(answer, dict):
        texts = answer.get("text", [])
        answer = texts[0] if texts else ""
    elif isinstance(answer, list):
        answer = answer[0] if answer else ""

    answer = str(answer).strip()
    return answer if answer else NO_ANSWER


if not TEST_PATH.is_file():
    raise FileNotFoundError(
        f"{TEST_PATH} was not found. Upload test_updated.jsonl to /tmp in Modal."
    )

test_records = []
with TEST_PATH.open("r", encoding="utf-8-sig") as test_file:
    for line_number, line in enumerate(test_file, 1):
        if not line.strip():
            continue
        try:
            item = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(
                f"Invalid JSON in {TEST_PATH} at line {line_number}: {error}"
            ) from error
        if not isinstance(item.get("question"), str) or not isinstance(item.get("context"), str):
            raise ValueError(f"Missing question/context in {TEST_PATH} at line {line_number}")
        test_records.append(item)

if LIMIT is not None:
    test_records = test_records[:LIMIT]

print(f"Loaded {len(test_records):,} QA records from {TEST_PATH}")
print(f"Incremental results will be saved to {RESULTS_PATH}")

correct_count = 0
answerable_correct = 0
answerable_total = 0
unanswerable_correct = 0
unanswerable_total = 0
error_count = 0
started_at = time.time()

with RESULTS_PATH.open("w", encoding="utf-8", newline="\n") as results_file:
    for index, item in enumerate(test_records, 1):
        question = item["question"].strip()
        expected = reference_answer(item)
        is_answerable = item.get("answerable") is not False
        error_message = None

        try:
            generated = ask_from_context(
                context=item["context"],
                question=question,
                max_new_tokens=MAX_NEW_TOKENS,
            )
        except Exception as error:
            generated = ""
            error_message = f"{type(error).__name__}: {error}"
            error_count += 1

        is_correct = error_message is None and normalize_answer(generated) == normalize_answer(expected)
        correct_count += int(is_correct)

        if is_answerable:
            answerable_total += 1
            answerable_correct += int(is_correct)
        else:
            unanswerable_total += 1
            unanswerable_correct += int(is_correct)

        result = {
            "index": index,
            "question": question,
            "expected_answer": expected,
            "generated_answer": generated,
            "answerable": is_answerable,
            "correct": is_correct,
            "error": error_message,
        }
        results_file.write(json.dumps(result, ensure_ascii=False) + "\n")
        results_file.flush()

        print("\n" + "=" * 100, flush=True)
        print(f"[{index:,}/{len(test_records):,}]", flush=True)
        print(f"Question  : {question}", flush=True)
        print(f"Expected  : {expected}", flush=True)
        print(f"Generated : {generated}", flush=True)
        print(f"Correct   : {is_correct}", flush=True)
        if error_message:
            print(f"Error     : {error_message}", flush=True)

elapsed = time.time() - started_at
total = len(test_records)
accuracy = (100.0 * correct_count / total) if total else 0.0
answerable_accuracy = (100.0 * answerable_correct / answerable_total) if answerable_total else 0.0
unanswerable_accuracy = (100.0 * unanswerable_correct / unanswerable_total) if unanswerable_total else 0.0

print("\n" + "=" * 100)
print("FINAL NORMALIZED EXACT-MATCH RESULTS")
print("=" * 100)
print(f"Correct              : {correct_count:,}/{total:,} ({accuracy:.2f}%)")
print(f"Incorrect            : {total - correct_count:,}/{total:,}")
print(f"Answerable correct   : {answerable_correct:,}/{answerable_total:,} ({answerable_accuracy:.2f}%)")
print(f"Unanswerable correct : {unanswerable_correct:,}/{unanswerable_total:,} ({unanswerable_accuracy:.2f}%)")
print(f"Inference errors     : {error_count:,}")
print(f"Elapsed seconds      : {elapsed:,.1f}")
print(f"Results file         : {RESULTS_PATH}")